In [1]:
# ══════════════════════════════════════════════════════════════
# 무신사 방식 "구매 평가" 웹 화면 (N-gram + Bag-of-Words + Gradio)
# 대상 파일: 상품평 엑셀 (필수 열: 상품평 / 있으면 사용하는 열: 카테고리, 상품명)
# ══════════════════════════════════════════════════════════════
#
# [전체 흐름]
#
#  ── 준비 단계 (셀 실행 시 1회) ─────────────────────────────────
#   0. 패키지 설치 : kiwipiepy(형태소 분석), openpyxl(엑셀 읽기), scikit-learn(CountVectorizer), gradio(웹 화면)
#   1. 형태소 분석 준비 : Kiwi 생성, 정도·부정 표현(KEEP) 정의, 토큰 → 단어 목록 변환 함수(to_words)
#   2. 표현 사전 : 강한 정도 2-gram 생성 함수(strong), 항목별 5단계 답과 N-gram 표현(ASPECTS)
#   3. BoW 함수 : 정해 둔 표현만 세는 BoW(count_phrases), 가장 긴 표현으로 답 고르기(choose_level)
#   4. 파일 분석 함수 : 엑셀 읽기 → 형태소 분석 → 띄어쓰기 문장 → 항목별 리뷰 답 (analyze_file)
#   5. 화면 함수 : 답별 비율표(rate_table) → 평가 화면 HTML(build_page), 2-gram 순위(top_bigrams)
#   6. 이벤트 함수 : 파일 선택 시(on_upload), 검색 버튼 클릭 시(on_search)
#   7. 웹 화면 구성·실행 : gr.Blocks로 화면 배치 → demo.launch()
#
#  ── 사용 단계 (웹 화면에서 반복) ───────────────────────────────
#   ① 엑셀 파일 선택  → on_upload → analyze_file → 분석 결과를 gr.State에 보관, 카테고리 목록 갱신
#   ② 조건 선택       → 카테고리, 리뷰 검색어, 2-gram 탐색 단어, 결과 표 항목·답 필터
#   ③ [검색 · 평가] 클릭(또는 검색어 입력 후 Enter) → on_search
#                     → 조건에 맞는 리뷰 선택 → 항목별 비율 계산 → 평가 화면 HTML
#                     → 탐색 단어가 들어간 2-gram 순위 계산
#   ④ 웹에 결과 표시  → 평가 화면(HTML) + 2-gram 순위 표 + 검색된 리뷰 표
#
#  ── 데이터 흐름 ────────────────────────────────────────────────
#   엑셀(.xlsx) → DataFrame(상품평) → Kiwi 토큰 → 단어 목록 → 띄어쓰기 문장("사이즈 너무 크다")
#      → CountVectorizer(vocabulary=표현, ngram_range=(1,3)) → 리뷰 × 표현 개수(BoW)
#      → 리뷰마다 나온 표현 중 가장 긴 N-gram의 답 선택 → 답별 리뷰 수·비율(%) → 막대 화면
#      → CountVectorizer(ngram_range=(2,2)) → 2-gram 전체 개수 → 탐색 단어가 든 2-gram 순위
#
# ══════════════════════════════════════════════════════════════

# 형태소 분석기, 엑셀 읽기 도구, BoW 도구, 웹 화면 도구 설치 (처음 한 번만 실행되면 됨)
%pip install -q kiwipiepy openpyxl scikit-learn gradio

# html 모듈: 화면에 넣을 글자 안의 <, >, & 같은 특수문자를 안전한 글자로 바꾸기 위해 가져옴
import html

# numpy: BoW 행렬의 열 합계를 1차원 배열로 바꾸기 위해 사용
import numpy as np
# pandas: 엑셀 데이터를 표(DataFrame)로 다루는 도구
import pandas as pd
# gradio: 파이썬 함수를 웹 화면(파일 선택, 버튼, 결과 표시)으로 만들어 주는 도구
import gradio as gr
# Kiwi: 한국어 형태소 분석기
from kiwipiepy import Kiwi
# CountVectorizer: 문서마다 단어(또는 N-gram)가 몇 번 나오는지 세어 '문서 × 표현' 개수 행렬(BoW)을 만드는 도구
from sklearn.feature_extraction.text import CountVectorizer


# ──────────────────────────────────────────────
# 1. 형태소 분석 준비
# ──────────────────────────────────────────────
# Kiwi 형태소 분석기 객체 생성
kiwi = Kiwi()

# 단어 목록에 그대로 남길 정도·부정·평가 표현 (N-gram에서 "너무 크다", "안 따뜻하다"를 만들기 위해 필요)
KEEP = {"너무", "매우", "아주", "엄청", "정말", "진짜", "완전", "많이",
        "조금", "좀", "약간", "살짝", "딱", "안", "못", "별로"}

# 형태소 분석 결과(토큰 목록)를 단어 목록으로 바꾸는 함수
#   동사·형용사·'않'·'없' → 뒤에 '다'를 붙임 (예: 커요 → 크다)
#   어근(XR)             → 뒤에 '하다'를 붙임 (예: 저렴 → 저렴하다)
#   명사, KEEP 표현       → 그대로 남김
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag
        if tag in {"VV", "VA"} or t.form in {"않", "없"}:
            words.append(t.form + "다")
        elif tag == "XR":
            words.append(t.form + "하다")
        elif tag in {"NNG", "NNP"} or t.form in KEEP:
            words.append(t.form)
    return words

# CountVectorizer 공통 설정
#   tokenizer=str.split : 띄어쓰기 기준으로만 자름
#   token_pattern=None  : 기본 정규식(2글자 이상 단어만)을 쓰지 않아 '좀', '딱', '안' 같은 한 글자 단어도 남김
#   lowercase=False     : 글자를 소문자로 바꾸지 않음
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)


# ──────────────────────────────────────────────
# 2. 표현 사전: 항목별 5단계 답과 N-gram 표현
# ──────────────────────────────────────────────
# 강한 정도 표현 목록
STRONG = ["너무", "매우", "아주", "엄청", "정말", "진짜", "완전", "많이"]

# words의 각 단어 앞에 STRONG 표현을 붙인 2-gram 목록을 만드는 함수
#   예: strong(["크다"]) → ["너무 크다", "매우 크다", ..., "많이 크다"]
def strong(words):
    return [f"{s} {w}" for s in STRONG for w in words]

# 항목별 5단계 답(화면에 나오는 순서)과 각 답으로 판단할 표현(1~3-gram) 목록
ASPECTS = {
    "사이즈": {
        "매우 작아요":   strong(["작다", "끼다"]),
        "조금 작아요":   ["작다", "끼다", "크다 주문", "크다 시키다"],
        "정사이즈예요":  ["정사이즈", "정 사이즈", "딱 맞다", "사이즈 맞다", "싸이즈 맞다", "치수 맞다", "품 맞다",
                         "사이즈 딱", "크다 않다", "작다 않다"],
        "조금 커요":     ["크다", "넉넉하다", "작다 주문", "작다 시키다"],
        "매우 커요":     strong(["크다", "넉넉하다"]),
    },
    "화면 대비 색감": {
        "매우 밝아요":   strong(["밝다", "연하다"]),
        "약간 밝아요":   ["밝다", "연하다"],
        "화면과 비슷해요": ["화면 같다", "사진 같다", "화면 동일", "실물 같다", "색상 같다", "색 같다", "화면 똑같다", "사진 똑같다"],
        "약간 어두워요": ["어둡다", "진하다"],
        "매우 어두워요": strong(["어둡다", "진하다"]),
    },
    "퀄리티": {
        "매우 별로예요": strong(["실망"]) + ["품질 너무 별로", "품질 정말 별로"],
        "별로예요":      ["실망", "불량", "보풀", "실밥", "품질 별로", "재질 별로", "원단 별로", "저렴하다 보이다", "싸다 보이다", "품질 안 좋다"],
        "괜찮아요":      ["품질 괜찮다", "재질 괜찮다", "원단 괜찮다", "괜찮다 품질", "품질 무난하다", "품질 보통"],
        "좋아요":        ["품질 좋다", "재질 좋다", "원단 좋다", "좋다 품질", "좋다 재질", "좋다 원단", "고급",
                         "보풀 없다", "보풀 안", "불량 없다", "실밥 없다"],
        "매우 좋아요":   [f"{n} {s} 좋다" for n in ["품질", "재질", "원단"] for s in STRONG],
    },
    "보온성": {
        "전혀 따뜻하지 않아요": ["따뜻하다 않다", "안 따뜻하다", "따듯하다 않다", "안 따듯하다"],
        "조금 따뜻해요":   ["조금 따뜻하다", "좀 따뜻하다", "약간 따뜻하다", "살짝 따뜻하다"],
        "적당해요":       ["따뜻하다 적당하다", "적당하다 따뜻하다", "보온 적당하다", "적당하다 보온"],
        "꽤 따뜻해요":     ["따뜻하다", "따듯하다", "보온 좋다", "포근하다"],
        "매우 따뜻해요":   strong(["따뜻하다", "따듯하다"]),
    },
}


# ──────────────────────────────────────────────
# 3. BoW로 표현 세기와 답 고르기
# ──────────────────────────────────────────────
# 한 항목의 표현만 리뷰마다 세는 함수 (정해 둔 표현만 세는 BoW)
#   sentences : 리뷰별 띄어쓰기 문장 목록
#   levels    : 한 항목의 답 딕셔너리 (예: ASPECTS["사이즈"])
#   반환값    : 행 = 리뷰, 열 = 표현, 값 = 등장 횟수인 DataFrame
def count_phrases(sentences, levels):
    phrases = [p for ps in levels.values() for p in ps]                        # 답별 표현 목록을 하나로 펼침
    cv = CountVectorizer(vocabulary=phrases, ngram_range=(1, 3), **opts)       # 1~3-gram 중 표현 목록에 있는 것만 셈
    X = cv.fit_transform(sentences)
    return pd.DataFrame(X.toarray(), columns=phrases)

# BoW 표의 한 행(리뷰 하나)에서 5단계 답 중 하나를 고르는 함수
#   1) 나온 표현이 없으면 None (이 항목을 말하지 않은 리뷰)
#   2) 나온 표현이 여러 개면 단어 수가 가장 많은 표현(가장 긴 N-gram)을 고름
#   3) 길이가 같으면 사전에서 먼저 나온 표현을 고름 (max는 같은 값일 때 앞의 것을 반환)
def choose_level(row, levels):
    found = [p for p in row.index if row[p] > 0]
    if not found:
        return None
    best = max(found, key=lambda p: len(p.split()))
    return next(level for level, ps in levels.items() if best in ps)


# ──────────────────────────────────────────────
# 4. 엑셀 파일을 읽고 모든 리뷰를 분석하는 함수 (파일 선택 시 1회 실행)
# ──────────────────────────────────────────────
# 엑셀 경로를 받아 리뷰별 띄어쓰기 문장과 항목별 답을 계산한 DataFrame을 반환
def analyze_file(path):
    # 상품평이 비어 있는 행은 제거하고, 상품평을 문자열로 변환
    df = pd.read_excel(path).dropna(subset=["상품평"]).reset_index(drop=True)
    df["상품평"] = df["상품평"].astype(str)
    # 카테고리·상품명 열이 없는 파일이면 빈 문자열 열을 만들어 둠
    for col in ["카테고리", "상품명"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str)

    # 리뷰 전체를 한 번에 형태소 분석하고, 단어를 띄어쓰기로 이어 붙인 문장을 저장
    #   예: "사이즈가 너무 커요" → "사이즈 너무 크다"
    df["문장"] = [" ".join(to_words(tokens)) for tokens in kiwi.tokenize(df["상품평"].tolist())]

    # 항목마다 BoW 표를 만들고, 리뷰마다 고른 답을 항목 이름 열로 저장
    for name, levels in ASPECTS.items():
        bow = count_phrases(df["문장"], levels)
        df[name] = bow.apply(choose_level, axis=1, levels=levels)
    return df


# ──────────────────────────────────────────────
# 5. 비율표, 평가 화면 HTML, 2-gram 순위 함수
# ──────────────────────────────────────────────
# 한 항목의 답별 리뷰 수와 비율(%) 표를 만드는 함수
#   value_counts()로 답별 개수를 세고, reindex로 사전 순서대로 정렬하며 없는 답은 0으로 채움
def rate_table(sub, name):
    count = sub[name].value_counts().reindex(list(ASPECTS[name]), fill_value=0)
    total = count.sum()
    percent = (count / total * 100).round().astype(int) if total else count * 0
    return pd.DataFrame({"리뷰 수": count, "비율": percent})

# 한 항목(5줄 막대)의 HTML을 만드는 함수
#   비율이 가장 높은 답은 파란 막대·굵은 글씨, 나머지는 회색
#   첫 줄 왼쪽에는 항목 이름, 둘째 줄 왼쪽에는 그 항목을 말한 리뷰 수를 표시
def aspect_html(name, table):
    total = int(table["리뷰 수"].sum())
    top = table["비율"].idxmax() if total else None
    rows = ""
    for i, (label, r) in enumerate(table.iterrows()):
        is_top = label == top
        text_style = "color:#111;font-weight:bold" if is_top else "color:#888"
        bar_color = "#2a5bf6" if is_top else "#bbb"
        title = {0: f"<b>{name}</b>", 1: f"<small>({total:,}건)</small>"}.get(i, "")
        rows += f"""
        <div style="display:grid;grid-template-columns:120px 150px 1fr 50px;align-items:center;font-size:13px;height:24px">
          <div style="color:#555;line-height:1.2">{title}</div>
          <div style="{text_style}">{label}</div>
          <div style="background:#ddd;height:6px;border-radius:3px;margin-right:12px">
            <div style="width:{r['비율']}%;background:{bar_color};height:6px;border-radius:3px"></div>
          </div>
          <div style="text-align:right;{text_style}">{r['비율']}%</div>
        </div>"""
    return f'<div style="border-top:1px solid #e2e2e2;padding:10px 0">{rows}</div>'

# 검색된 리뷰 묶음(sub)으로 평가 화면 HTML 전체를 만드는 함수
def build_page(sub, title):
    groups = "".join(aspect_html(name, rate_table(sub, name)) for name in ASPECTS)
    return f"""
    <div style="font-family:sans-serif;background:#f4f4f4;color:#222;padding:20px 24px;max-width:820px;border-radius:6px">
      <div style="font-weight:bold;font-size:15px;margin-bottom:12px">{html.escape(title)} · 리뷰 {len(sub):,}건으로 분석한 평가예요</div>
      {groups}
      <div style="color:#999;font-size:11px;margin-top:8px">* 리뷰 글에 나온 표현(N-gram)을 세어 추정한 결과입니다. 항목을 말한 리뷰만 계산했습니다.</div>
    </div>"""

# 검색된 리뷰의 문장에서 2-gram을 모두 세고, target 단어가 들어간 2-gram을 많은 순서로 n개 돌려주는 함수
#   target이 비어 있으면 전체 2-gram 상위 n개를 돌려줌
#   target 비교는 2-gram을 띄어쓰기로 나눈 단어와 '정확히 같은지'로 판단
def top_bigrams(sentences, target, n=15):
    sentences = [s for s in sentences if s.strip()]
    if not sentences:
        return pd.DataFrame(columns=["2-gram", "등장 횟수"])
    cv = CountVectorizer(ngram_range=(2, 2), **opts)
    try:
        X = cv.fit_transform(sentences)
    except ValueError:                                                          # 2-gram이 하나도 없을 때
        return pd.DataFrame(columns=["2-gram", "등장 횟수"])
    counts = pd.Series(np.asarray(X.sum(axis=0)).ravel(), index=cv.get_feature_names_out())
    if target:
        counts = counts[[target in gram.split() for gram in counts.index]]
    counts = counts.sort_values(ascending=False).head(n)
    return pd.DataFrame({"2-gram": counts.index, "등장 횟수": counts.values})


# ──────────────────────────────────────────────
# 6. Gradio 이벤트 함수
# ──────────────────────────────────────────────
# 검색 결과 표에 보여 줄 열 순서
RESULT_COLS = ["카테고리", "상품명", "상품평"] + list(ASPECTS)

# 결과 표의 '답' 선택지를 항목에 맞게 바꾸는 함수 (항목 드롭다운이 바뀔 때 실행)
def on_aspect_change(aspect):
    choices = ["전체"] if aspect == "전체" else ["전체"] + list(ASPECTS[aspect])
    return gr.update(choices=choices, value="전체")

# [파일 선택 시] 엑셀을 분석해 상태(state)에 저장하고, 카테고리 선택지와 안내 문구를 돌려줌
def on_upload(file):
    if file is None:
        return None, gr.update(choices=["전체"], value="전체"), "엑셀 파일을 선택하세요."
    path = file if isinstance(file, str) else file.name                       # Gradio 버전에 따라 경로 문자열 또는 파일 객체
    df = analyze_file(path)
    cats = ["전체"] + sorted(c for c in df["카테고리"].unique() if c)
    said = ", ".join(f"{name} {df[name].notna().sum():,}건" for name in ASPECTS)
    msg = f"분석 완료: 리뷰 {len(df):,}건 (항목을 말한 리뷰 → {said})"
    return df, gr.update(choices=cats, value="전체"), msg

# [검색 버튼 클릭 시] 조건에 맞는 리뷰를 골라 평가 화면, 2-gram 순위, 리뷰 표, 안내 문구를 돌려줌
def on_search(df, category, keyword, target, aspect, level):
    empty_reviews = pd.DataFrame(columns=RESULT_COLS)
    empty_bigrams = pd.DataFrame(columns=["2-gram", "등장 횟수"])
    if df is None:
        return "<p>먼저 엑셀 파일을 선택하세요.</p>", empty_bigrams, empty_reviews, ""

    sub = df
    if category and category != "전체":
        sub = sub[sub["카테고리"] == category]                                # 카테고리 필터
    keyword = (keyword or "").strip()
    if keyword:
        # 검색어가 상품명 또는 상품평에 들어 있는 리뷰만 남김 (정규식 해석 없이 글자 그대로 검색)
        hit = (sub["상품명"].str.contains(keyword, case=False, regex=False)
               | sub["상품평"].str.contains(keyword, case=False, regex=False))
        sub = sub[hit]
    if sub.empty:
        return "<p>검색 결과가 없습니다.</p>", empty_bigrams, empty_reviews, "검색 결과 0건"

    # 평가 화면: 선택한 카테고리와 검색어를 제목으로 사용
    title = " / ".join(x for x in [category if category != "전체" else "전체 카테고리", keyword] if x)
    page = build_page(sub, title)

    # 2-gram 순위: 탐색 단어(형태소 분석 후 모양, 예: 크다)가 들어간 2-gram
    target = (target or "").strip()
    bigrams = top_bigrams(sub["문장"].tolist(), target)

    # 리뷰 표: 항목·답 필터 적용 (항목만 고르면 그 항목을 말한 리뷰 전체)
    table = sub
    if aspect != "전체":
        table = table[table[aspect].notna()] if level == "전체" else table[table[aspect] == level]
    table = table[RESULT_COLS].copy()
    table["상품평"] = table["상품평"].str.replace("\n", " ", regex=False).str.strip()
    table[list(ASPECTS)] = table[list(ASPECTS)].fillna("-")                   # 말하지 않은 항목은 '-'로 표시

    info = (f"검색된 리뷰 {len(sub):,}건 · 표에 표시 {len(table):,}건"
            f" (항목: {aspect}, 답: {level}) · 2-gram 탐색 단어: {target or '없음(전체 상위)'}")
    return page, bigrams, table, info


# ──────────────────────────────────────────────
# 7. Gradio 웹 화면 구성과 실행
# ──────────────────────────────────────────────
with gr.Blocks(title="구매 평가 화면 (N-gram + BoW)") as demo:
    gr.Markdown("## 👕 구매한 회원들이 평가했어요 (N-gram + Bag-of-Words)")
    state = gr.State(None)                                                     # 분석된 DataFrame을 보관하는 상태 값

    # 전체 흐름 안내 (클릭하면 펼쳐지는 영역)
    with gr.Accordion("📋 전체 흐름 보기", open=False):
        gr.Markdown(
            "| 단계 | 화면에서 할 일 | 내부 처리 | 결과 |\n"
            "|---|---|---|---|\n"
            "| ① 파일 선택 | 엑셀(.xlsx) 선택 | 엑셀 읽기 → Kiwi 형태소 분석 → 띄어쓰기 문장 → 항목별 BoW → 리뷰별 답 | 상태 창에 리뷰 수, 항목별 언급 수 |\n"
            "| ② 조건 선택 | 카테고리, 리뷰 검색어, 2-gram 탐색 단어, 결과 표 항목·답 | — | — |\n"
            "| ③ 검색 · 평가 | 버튼 클릭 또는 Enter | 조건에 맞는 리뷰 선택 → 답별 비율 → 2-gram 순위 | 평가 화면 + 2-gram 표 + 리뷰 표 |\n\n"
            "**N-gram**: 붙어 있는 단어 N개를 한 묶음으로 봄 (`크다` → `너무 크다`로 정도 구분)  \n"
            "**BoW**: 리뷰마다 표현이 몇 번 나왔는지 센 표 (`CountVectorizer`)  \n"
            "**답 고르기**: 나온 표현 중 가장 긴 N-gram의 답, 길이가 같으면 사전에서 먼저 나온 답"
        )

    with gr.Row():
        file_in = gr.File(label="① 엑셀 파일 선택 (.xlsx)", file_types=[".xlsx"])
        with gr.Column():
            status = gr.Textbox(label="상태", value="엑셀 파일을 선택하세요.", interactive=False)
            category = gr.Dropdown(label="② 카테고리", choices=["전체"], value="전체")
            keyword = gr.Textbox(label="③ 리뷰 검색어 (상품명·상품평, 비우면 전체)", placeholder="예: 바지, 패딩, 니트")
            target = gr.Textbox(label="④ 2-gram 탐색 단어 (형태소 분석 후 모양, 비우면 전체 상위)", placeholder="예: 크다, 작다, 따뜻하다, 품질")
            with gr.Row():
                aspect = gr.Dropdown(label="⑤ 결과 표 항목", choices=["전체"] + list(ASPECTS), value="전체")
                level = gr.Dropdown(label="⑥ 결과 표 답", choices=["전체"], value="전체")
            btn = gr.Button("🔍 검색 · 평가", variant="primary")

    page_out = gr.HTML()                                                       # 평가 화면이 표시되는 영역
    info_out = gr.Markdown()                                                   # 검색 건수 안내 문구
    with gr.Row():
        bigram_out = gr.Dataframe(label="2-gram 순위", scale=1)                # 탐색 단어가 들어간 2-gram 표
        table_out = gr.Dataframe(label="검색된 리뷰", wrap=True, scale=3)      # 검색된 리뷰와 항목별 답 표

    # 파일을 선택하면 on_upload 실행 → 상태·카테고리 선택지·안내 문구 갱신
    file_in.change(on_upload, inputs=file_in, outputs=[state, category, status])
    # 결과 표 항목을 바꾸면 그 항목의 5단계 답으로 '답' 선택지를 바꿈
    aspect.change(on_aspect_change, inputs=aspect, outputs=level)
    # 버튼 클릭 또는 검색어·탐색 단어 입력 후 Enter → on_search 실행 → 화면·표·안내 문구 갱신
    search_inputs = [state, category, keyword, target, aspect, level]
    search_outputs = [page_out, bigram_out, table_out, info_out]
    btn.click(on_search, inputs=search_inputs, outputs=search_outputs)
    keyword.submit(on_search, inputs=search_inputs, outputs=search_outputs)
    target.submit(on_search, inputs=search_inputs, outputs=search_outputs)

# 웹 서버 실행 (노트북 셀 아래에 화면이 표시되고, 출력되는 주소로 브라우저에서도 열 수 있음)
demo.launch()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 8.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.6 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6924de5274a2c8e53.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
